# Train GridCRNN

Notebook này chỉ train và ghi weights. Inference nằm trong `baseline_TACVU2_improve_1.ipynb`.


In [ ]:
import json
from pathlib import Path

SOURCE_NOTEBOOK = Path.cwd() / 'baseline_TACVU2_improve_1.ipynb'
if not SOURCE_NOTEBOOK.is_file():
    raise FileNotFoundError(f'Không thấy notebook nguồn: {SOURCE_NOTEBOOK}')

source = json.loads(SOURCE_NOTEBOOK.read_text(encoding='utf-8'))
for cell_index in (2, 6, 8, 10):
    exec(''.join(source['cells'][cell_index]['source']), globals())

print(f'[train] loaded shared definitions from {SOURCE_NOTEBOOK.name}')


In [ ]:
MODEL_DIR = RUNS / 'model_a'
EPOCHS = 8
BATCH_SIZE = 48
LEARNING_RATE = 8e-4
MAX_SAMPLES = 1000

set_seed(SEED)
torch.manual_seed(SEED)
print(f'[train] seed={SEED} | epochs={EPOCHS} | batch={BATCH_SIZE} | max_samples={MAX_SAMPLES}')

train_records = load_manifest(TRAIN_DIR)
samples = collect_cell_samples(TRAIN_DIR, train_records, MAX_SAMPLES)
if not samples:
    raise ValueError('Không trích được ô nào từ tập train')
print(f'[train] {len(train_records)} tài liệu -> {len(samples)} ô dùng để học')

alphabet = Alphabet.fit([sample.text for sample in samples])
dataset = CellDataset(samples, alphabet)
loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_cells,
    generator=torch.Generator().manual_seed(MODEL_SEED),
)
model = GridCRNN(len(alphabet.characters) + 1).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)

model.train()
for epoch in range(1, EPOCHS + 1):
    total = count = 0
    started = time.time()
    for images, targets, target_lengths, widths in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        logits = model(images)
        input_lengths = torch.clamp((widths + 3) // 4, max=logits.shape[1]).to(dtype=torch.long)
        loss = criterion(logits.transpose(0, 1), targets, input_lengths, target_lengths)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += float(loss.detach().cpu()) * len(images)
        count += len(images)
    print(f'[train] epoch {epoch}/{EPOCHS} | loss={total / max(1, count):.5f} | {time.time() - started:.1f}s')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
torch.save({
    'state_dict': model.state_dict(), 'characters': alphabet.characters, 'seed': MODEL_SEED,
    'samples': len(samples), 'epochs': EPOCHS,
}, MODEL_DIR / 'grid_crnn.pt')
print(f"[train] saved {MODEL_DIR / 'grid_crnn.pt'}")
